# GARCH(1,1) conditional-volatility experiment

This experiment models 100 times the log return of TRM with a constant conditional mean and GARCH(1,1) variance. It forecasts cumulative h-step return variance for h=1,...,10. It is not a replacement for the persistence or ARIMA level forecasts.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from trm_forecast.evaluation.garch import (
    block_bootstrap_coverage, evaluate_garch, evaluate_variance_benchmark
)
from trm_forecast.evaluation.protocol import PartitionSpec, TemporalEvaluationProtocol
from trm_forecast.models.garch import log_returns

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
config = yaml.safe_load((ROOT / 'configs/base.yaml').read_text())
frame = pd.read_csv(ROOT / 'data/interim/trm_normalized.csv', parse_dates=['date'])
series = frame.set_index('date')['trm'].sort_index()
evaluation = config['evaluation']
specs = tuple(PartitionSpec(name, values['start'], values['end']) for name, values in evaluation['partitions'].items())
result = TemporalEvaluationProtocol(evaluation['as_of_date'], tuple(evaluation['horizons']), specs).split(series)
pd.DataFrame([summary.__dict__ for summary in result.summaries])

## Training-only stationarity and ARCH diagnostics

In [ ]:
garch = config['garch']
train_returns = log_returns(result.partitions['train'], scale=garch['scale'])
print('observations:', len(train_returns), 'mean:', train_returns.mean(), 'std:', train_returns.std(ddof=1))
print('Engle ARCH LM:', het_arch(train_returns, nlags=10)[:2])
acorr_ljungbox(train_returns ** 2, lags=[5, 10], return_df=True)

## Validation

The variance target is the squared cumulative scaled log return from origin to target. QLIKE evaluates the forecast variance. The level interval is a plug-in normal approximation using the summed recursive GARCH variances; it is reported separately from level MAE/RMSE. Coverage uncertainty uses a moving-block bootstrap with block length 10 to reflect overlapping windows.

In [ ]:
horizons = tuple(evaluation['horizons'])
validation_predictions, validation_metrics = evaluate_garch(
    result, horizons, ('validation',), garch['p'], garch['q'], garch['min_history'],
    garch['scale'], garch['maxiter'], garch['refit_frequency'], garch['confidence_level']
)
validation_metrics
validation_benchmark_predictions, validation_benchmark_metrics = evaluate_variance_benchmark(
    result, horizons, ('validation',), garch['scale']
)
validation_benchmark_metrics
validation_metrics.assign(
    coverage_lower=validation_predictions.groupby('horizon')['covered'].apply(
        lambda values: block_bootstrap_coverage(values.to_numpy(), block_length=10)[0]
    ).to_numpy(),
    coverage_upper=validation_predictions.groupby('horizon')['covered'].apply(
        lambda values: block_bootstrap_coverage(values.to_numpy(), block_length=10)[1]
    ).to_numpy(),
)

## Frozen final test evaluation

No test result is used to select the specification. The configured GARCH(1,1) specification is evaluated once on the untouched test partition.

In [ ]:
test_predictions, test_metrics = evaluate_garch(
    result, horizons, ('test',), garch['p'], garch['q'], garch['min_history'],
    garch['scale'], garch['maxiter'], garch['refit_frequency'], garch['confidence_level']
)
test_metrics
test_benchmark_predictions, test_benchmark_metrics = evaluate_variance_benchmark(
    result, horizons, ('test',), garch['scale']
)
test_benchmark_metrics
test_metrics.assign(
    coverage_lower=test_predictions.groupby('horizon')['covered'].apply(
        lambda values: block_bootstrap_coverage(values.to_numpy(), block_length=10)[0]
    ).to_numpy(),
    coverage_upper=test_predictions.groupby('horizon')['covered'].apply(
        lambda values: block_bootstrap_coverage(values.to_numpy(), block_length=10)[1]
    ).to_numpy(),
)